<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/K21_Konsult_51.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/parashar21/main/images/khona21-v0.png) <br>


---



[Prithwis Mukerjee](http://www.yantrajaal.com) ||
Website - [Parashar21](https://parashar21.blogspot.com)<br>

# Khona21 — Interpretive Jyotiṣa

**Khona21** is an experimental AI-assisted Jyotiṣa interpretation platform, developed as a companion to **Parashar21**, which performs the astronomical calculations underlying traditional Indian astrology.

Named after the legendary Bengali astrologer Khona, the platform combines precomputed Rashi and Navamsha charts, Vimshottari Dasha schedules and planetary transits (Gochar) with a curated corpus of Jyotiṣa literature, currently comprising works by **James Braha** and **Harihar Majumdar**.

Rather than relying on hard-coded interpretive rules, Khona21 employs **source-grounded, multipass LLM reasoning** to examine planetary configurations, yogas, cancellations, exceptions and conflicting interpretations. An initial assessment establishes the horoscope's principal characteristics, while subsequent consultations investigate specific questions using the original reference texts and relevant astrological evidence.

This notebook, **k21_KhonaPlus**, implements the consultation workflow, integrating deterministic calculations from Parashar21 with the interpretive capabilities of modern Large Language Models.

**Khona21 explores the convergence of traditional Jyotiṣa knowledge and modern artificial intelligence, combining computational precision with source-grounded reasoning to produce systematic, transparent and auditable astrological interpretations.**

In [1]:
print("\033[1m"+'জন্ম-মৃত্যু-বিবাহ, তিন না জানেন বরাহ।'+"\033[0m")
!TZ='Asia/Kolkata' date

জন্ম-মৃত্যু-বিবাহ, তিন না জানেন বরাহ।
Sun Nov 15 12:42:04 PM IST 2026


#Setup, Installs, Imports

In [2]:
!pip -q install pypandoc
!pip -q install python-docx
from IPython.display import display, Markdown
from datetime import datetime
import pytz


!wget -O k21core.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21core.py
!wget -O k21fileops.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21fileops.py
!wget -O khona.png -q https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/khana-sign-0.png

from k21core import *
from k21fileops import *

In [3]:
# ------------------------------------------------
# required for multiple imports of the same module
# ------------------------------------------------
import k21core
import k21fileops
import importlib

importlib.reload(k21core)
importlib.reload(k21fileops)
from k21core import *
from k21fileops import *

In [4]:
# ------------------------------------------------
# Set Native ID
# ------------------------------------------------
Native_id = "TopGun"

In [5]:
# ------------------------------------------------
# Use OpenAI
# ------------------------------------------------
provider = "openai"

#model = "gpt-5.6-luna"
#model = "gpt-6-luna"
model = "gpt-6.1-sol"
#model = "gpt-6-astra"

# ------------------------------------------------
# Use Claude
# ------------------------------------------------
#!pip install -q anthropic

#provider = "claude"
#model = "claude-haiku-5-5"
#model = "claude-sonnet-5-5"


###Access Control
This notebook accesses Google Drive to access files and uses the OpenAI API. Hence it can be used ONLY from the userid of the creator. Others will have to create their own access paths

In [6]:
# ------------------------------------------------
# Authenticate wit LLM provider using API keys stored as Colab Secret
# ------------------------------------------------
client = authenticateLLM(provider=provider)

OPENAI | User: Calcutta | Email: prithwis@yantrajaal.com | ID: user-sNAljKNmBf2igtAV2kUuqhoF


In [7]:
# ------------------------------------------------
# Locate necessary file and set up paths to the same
# ------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
#ls /content/drive/MyDrive/Prithwis/Parashar21/K21

K21_HOME = "/content/drive/MyDrive/Prithwis/Parashar21/K21"

# ------------------------------------------------

BRAHA   = f"{K21_HOME}/Corpus/BR_Curated_22.txt"
HARIHAR = f"{K21_HOME}/Corpus/HM_Clean_07.txt"

#RAMAN = f"{K21_HOME}/Corpus/Raman_08.md"
#YOGS = f"{K21_HOME}/Corpus/300_Astrology_Yogas.md"

# ------------------------------------------------

#FIRSTPASS = f"{K21_HOME}/FirstPass/"
#QUERY_DIR = f"{K21_HOME}/Query/"



Mounted at /content/drive


## Load Files

In [8]:
#
# Following files from Parashar21/Kompute need to be uploaded here
#

import zipfile
from google.colab import files

# Upload ZIP file
uploaded = files.upload()
zipName = next(iter(uploaded))

# Extract all files into current directory
with zipfile.ZipFile(zipName, 'r') as zf:
    zf.extractall()

print("Extracted:", zipName)

CHART = "/content/Chart.txt"
NAVAMSA = "/content/Navamsa_Chart.txt"
DASHA = "/content/Dasha.txt"
GOCHAR = "/content/Gochar_Chart.txt"
GOCHAR_EVENTS = "/content/Transit.txt"

Saving Kompute_261115_092624.zip to Kompute_261115_092624.zip
Extracted: Kompute_261115_092624.zip


# LLM Access




In [9]:
# ------------------------------------------------
# Define the ROLE
# ------------------------------------------------
ROLEFile = f"{K21_HOME}/Roles/khona261029A.txt"
khona21 = readText(ROLEFile)

/content/drive/MyDrive/Prithwis/Parashar21/K21/Roles/khona261029A.txt | 6,774 characters


In [10]:
# ------------------------------------------------
# Set the Context
# ------------------------------------------------
#BaseCONTEXT = buildContext(BRAHA, HARIHAR, CHART, NAVAMSA, DASHA)

BaseCONTEXT = getContext(
    BRAHA=BRAHA,
    HARIHAR=HARIHAR,
    CHART=CHART,
    NAVAMSA=NAVAMSA,
    DASHA=DASHA
)

len(BaseCONTEXT)

944654

## the FirstPass

In [12]:
# ------------------------------------------------
# FIRST PASS
#
# ------------------------------------------------
Instructions = f"{K21_HOME}/Query/FirstPassInstructions.txt"
FirstPass_Instruction = readText(Instructions)

/content/drive/MyDrive/Prithwis/Parashar21/K21/Query/FirstPassInstructions.txt | 1,111 characters


In [14]:
# ------------------------------------------------
# FirstPass: Load from cache or generate and save
# ------------------------------------------------

from pathlib import Path

FIRSTPASS = f"{K21_HOME}/FirstPass/"
cachedFiles = sorted(Path(FIRSTPASS).glob(f"{Native_id}_*.md"))

if cachedFiles:
    # Existing FirstPass
    MD_FILE = str(cachedFiles[-1])
    FirstPass = Path(MD_FILE).read_text(encoding="utf-8")
    print(f"FirstPass loaded: {MD_FILE}")

else:
    # Generate new FirstPass
    QUERY = buildQuery(BaseCONTEXT, FirstPass_Instruction)

    FirstPass = callLLM(
        client=client,
        role=khona21,
        query=QUERY,
        model=model,
        provider=provider,
        effort="medium"
    )

    # Save locally, then copy to Google Drive
    MD_FILE = saveAnswer(model, FirstPass)
    _ = saveFirstPass(MD_FILE, Native_id, FIRSTPASS)

    print(f"FirstPass generated and saved: {MD_FILE}")

OPENAI | gpt-6.1-sol | Effort: medium | In: 222,676 | Cached: 0 | Out: 2,208 | Reasoning: 334 | Time: 56.7s
Saved: /content/K21_261115_125249_gpt-6.1-sol.md
FirstPass saved: /content/drive/MyDrive/Prithwis/Parashar21/K21/FirstPass/TopGun_261115_gpt-6.1-sol.md
FirstPass generated and saved: /content/K21_261115_125249_gpt-6.1-sol.md


In [16]:
# ------------------------------------------------
# Display FirstPass (both cases)
# ------------------------------------------------
#display(Markdown(FirstPass))

## Add Gochar and Transit Info

In [17]:
Gochar = readText(GOCHAR)
Transits = readText(GOCHAR_EVENTS)

/content/Gochar_Chart.txt | 4,584 characters
/content/Transit.txt | 5,494 characters


In [18]:
CONTEXT = augContext(
    BaseCONTEXT,
    FIRSTPASS=FirstPass,
    GOCHAR=Gochar,
    TRANSITS=Transits
)

len(CONTEXT)

963731

## Konsultation

In [22]:
CurrentQuestion = readText(f"{K21_HOME}/Query/Gochar30days.txt")

/content/drive/MyDrive/Prithwis/Parashar21/K21/Query/Gochar30days.txt | 2,603 characters


In [23]:
# ------------------------------------------------
# Assemble the query
# Call the LLM
# ------------------------------------------------

#model = "gpt-6.1-sol"

QUERY = buildQuery(CONTEXT, CurrentQuestion)

answer = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)

OPENAI | gpt-6.1-sol | Effort: medium | In: 228,340 | Cached: 0 | Out: 4,560 | Reasoning: 516 | Time: 107.3s


In [24]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(model, answer)
# ------------------------------------------------
# Convert markdown to docx for human readability
# Second Parameter is identifier of native, can be replaced by id
# ------------------------------------------------
DOCX_FILE = markdownToDocx(MD_FILE, Native_id, CurrentQuestion, provider,model)

Saved: /content/K21_261115_130635_gpt-6.1-sol.md
Saved: /content/K21_261115_130635_gpt-6.1-sol.docx


## Check Credit Status
https://platform.openai.com/settings/organization/billing/credit-grants

https://platform.openai.com/usage

https://platform.claude.com/settings/billing

https://platform.claude.com/usage


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)